# Spatial Regression: Mean Structure and Correlated Errors

Spatial dependence can remain after a regression mean has been fitted. This notebook simulates a linear model with spatially correlated errors and contrasts OLS with GLS when the simulation covariance is known.

Knowing the covariance is unrealistic in practice; it is used here only to isolate the statistical consequence of dependence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.spatial.distance import cdist

rng = np.random.default_rng(1)
n = 180
coords = rng.uniform(0, 1, size=(n, 2))
x = rng.normal(size=n)
D = cdist(coords, coords)

Sigma = 0.8*np.exp(-D/0.18) + 0.25*np.eye(n)
error = rng.multivariate_normal(np.zeros(n), Sigma)
y = 1.0 + 2.0*x + error
X = sm.add_constant(x)

ols = sm.OLS(y, X).fit()
gls = sm.GLS(y, X, sigma=Sigma).fit()

print("OLS params:", np.round(ols.params, 3), "SE:", np.round(ols.bse, 3))
print("GLS params:", np.round(gls.params, 3), "SE:", np.round(gls.bse, 3))


Notice especially the intercept uncertainty: spatially correlated errors can contain broad low-frequency variation that makes an overall level less precisely estimated than an iid OLS formula suggests.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(coords[:,0], coords[:,1], c=ols.resid, s=45)
fig.colorbar(sc, ax=ax, label="OLS residual")
ax.set(title="OLS residuals retain spatial structure", xlabel="x", ylabel="y")
plt.show()


In [ ]:
# Build row-standardized 6-nearest-neighbor weights and compute Moran's I.
D2 = D.copy()
np.fill_diagonal(D2, np.inf)
k = 6
nbr = np.argpartition(D2, kth=k-1, axis=1)[:, :k]
W = np.zeros((n,n))
W[np.repeat(np.arange(n), k), nbr.ravel()] = 1
W = np.maximum(W, W.T)
W = W / W.sum(axis=1, keepdims=True)

def moran_i(v, W):
    z = v - v.mean()
    return len(v)/W.sum() * (z @ W @ z)/(z @ z)

print("Moran's I of OLS residuals:", round(moran_i(ols.resid, W), 3))


A significant residual spatial pattern does not by itself tell us whether to use a spatial error model, a spatial lag model, a continuous covariance model, or a richer mean function. The scientific mechanism and spatial data type determine that choice.

Also keep **prediction**, **association**, and **causation** separate: modeling residual spatial dependence does not make a coefficient causal.